In [31]:
import tkinter as tk
from tkinter import messagebox
import random
import heapq
import time
from collections import deque
import copy
GRID_SIZE = 15
BLOCK_PROB = 0.32
CHECKPOINTS = 1
HINTS_TOTAL = 5
AUTO_STEP_DELAY_MS = 90
CELL_PIXEL = 56
SMOOTH_STEP = 8
HINT_SHOW_MS = 1000

PLAYER_EMOJI = "😎"
GOAL_EMOJI = "💎"
CHECK_EMOJI = "🎯"
BLOCK_EMOJI = "🔥"
PATH_EMOJI = "🟦"

BG = "#071020"
CELL_BG = "#08121a"
TEXT = "#e6f7ff"
GOAL_BG = "#ffd166"
CHECK_BG = "#c084fc"
PATH_BG = "#00e5ff"
BLOCK_BG = "#3b0b0b"
VISITED_BG = "#0b2a2a"
CHECKED_BG = "#064e3b" 


def neighbors(r, c, grid):
    """Return 4-neighborhood walkable neighbors (up/down/left/right)."""
    for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < len(grid) and 0 <= nc < len(grid[0]) and grid[nr][nc] == 0:
            yield (nr, nc)

def reconstruct_path(came_from, end):
    """Reconstruct path from came_from dict (end -> start)."""
    path = []
    cur = end
    while cur in came_from:
        path.append(cur)
        cur = came_from[cur]
    path.reverse()
    return path

def bfs_once(start, goal, grid):
    """BFS that returns (path, visited_nodes). start/goal can be tuples or lists."""
    start = tuple(start); goal = tuple(goal)
    q = deque([start])
    came_from = {}
    visited = set([start])
    visited_nodes = []
    while q:
        node = q.popleft()
        visited_nodes.append(node)
        if node == goal:
            return reconstruct_path(came_from, goal), visited_nodes
        for nb in neighbors(*node, grid):
            if nb not in visited:
                visited.add(nb)
                came_from[nb] = node
                q.append(nb)
    return None, visited_nodes

def a_star_once(start, goal, grid):
    """A* (Manhattan) that returns (path, visited_nodes)."""
    start = tuple(start); goal = tuple(goal)
    def h(a,b):
        return abs(a[0]-b[0]) + abs(a[1]-b[1])
    open_set = []
    heapq.heappush(open_set, (0, start))
    came_from = {}
    g = {start: 0}
    visited_nodes = []
    seen = set()
    while open_set:
        _, current = heapq.heappop(open_set)
        if current in seen:
            continue
        seen.add(current)
        visited_nodes.append(current)
        if current == goal:
            return reconstruct_path(came_from, goal), visited_nodes
        for nb in neighbors(*current, grid):
            tentative = g[current] + 1
            if tentative < g.get(nb, float('inf')):
                came_from[nb] = current
                g[nb] = tentative
                f = tentative + h(nb, goal)
                heapq.heappush(open_set, (f, nb))
    return None, visited_nodes


def carve_corridor(grid, a, b):
    """Make a straight corridor connecting two points (simple Manhattan carve)."""
    (r1,c1),(r2,c2) = a,b
    r,c = r1,c1
    grid[r][c] = 0
    while r != r2:
        r += 1 if r2 > r else -1
        grid[r][c] = 0
    while c != c2:
        c += 1 if c2 > c else -1
        grid[r][c] = 0

def generate_solvable_grid(size, block_prob, checkpoints):
    """Generate a grid with random blocks but carve corridors so start->checkpoints->goal are reachable."""
    attempts = 0
    while True:
        attempts += 1
        grid = [[1 if random.random() < block_prob else 0 for _ in range(size)] for _ in range(size)]
        free = [(r,c) for r in range(size) for c in range(size) if grid[r][c]==0]
        if len(free) < (checkpoints + 3):
            continue
        start = random.choice(free)
        free2 = [p for p in free if p != start]
        goal = random.choice(free2)
        if goal == start:
            continue
        random.shuffle(free2)
        cps = []
        for p in free2:
            if p != start and p != goal:
                cps.append(p)
            if len(cps) >= checkpoints:
                break
        waypoints = [start] + cps + [goal]
        ok = True
        for i in range(len(waypoints)-1):
            p, _ = bfs_once(waypoints[i], waypoints[i+1], grid)
            if not p:
                carve_corridor(grid, waypoints[i], waypoints[i+1])
                p2, _ = bfs_once(waypoints[i], waypoints[i+1], grid)
                if not p2:
                    ok = False
                    break
        if ok:
            return grid, start, cps, goal
        if attempts > 300:
            block_prob = max(0.06, block_prob - 0.03)


class EmojiRushSmooth:
    """Main game class: grid rendering, player movement, hints, A* assist, scoring."""
    def __init__(self, master):
        self.master = master
        master.title("Emoji Rush")
        master.config(bg=BG)

        # runtime state
        self.level = 1
        self.animating = False
        self.total_stars = 0

        self.init_level()

    def init_level(self):
        """Initialize a random solvable level and UI."""
        level_checkpoints = min(self.level, 5)
        self.grid, self.start, self.checkpoints, self.goal = generate_solvable_grid(
            GRID_SIZE, BLOCK_PROB, level_checkpoints
        )

        # Keep original checkpoint order for shortest-path calc
        self.original_checkpoints = copy.deepcopy(self.checkpoints)

        self.player = list(self.start)
        self.hints_left = HINTS_TOTAL
        self.steps = 0
        self.level_over = False
        self.start_time = time.time()

        self.build_ui()
        self.update_timer()


    def build_ui(self):
        """Build or rebuild the entire UI for the current grid/level."""
        for w in self.master.winfo_children():
            w.destroy()

        top = tk.Frame(self.master, bg=BG)
        top.pack(pady=6)

        self.level_label = tk.Label(
            top,
            text=f"Level {self.level}  🎯 {len(self.checkpoints)} checkpoints    ⭐ Total: {self.total_stars}",
            font=("Helvetica", 14, "bold"),
            fg=TEXT,
            bg=BG
        )
        self.level_label.grid(row=0, column=0, columnspan=4, sticky="w", padx=6)

        self.info_steps = tk.Label(top, text=f"Steps: {self.steps}", fg=TEXT, bg=BG)
        self.info_steps.grid(row=1, column=0, padx=8)
        self.info_hints = tk.Label(top, text=f"Hints: {self.hints_left}", fg=TEXT, bg=BG)
        self.info_hints.grid(row=1, column=1, padx=8)
        self.info_timer = tk.Label(top, text="Time: 0s", fg=TEXT, bg=BG)
        self.info_timer.grid(row=1, column=2, padx=8)
        tk.Button(top, text="Next Level", command=self.next_level, width=10).grid(row=1, column=3, padx=8)

        canvas_w = GRID_SIZE * CELL_PIXEL
        canvas_h = GRID_SIZE * CELL_PIXEL
        self.canvas = tk.Canvas(self.master, width=canvas_w, height=canvas_h, bg=BG, highlightthickness=0)
        self.canvas.pack(padx=8, pady=6)

        self.rects = [[None]*GRID_SIZE for _ in range(GRID_SIZE)]
        self.texts = [[None]*GRID_SIZE for _ in range(GRID_SIZE)]
        for r in range(GRID_SIZE):
            for c in range(GRID_SIZE):
                x1 = c*CELL_PIXEL; y1 = r*CELL_PIXEL
                x2 = x1 + CELL_PIXEL; y2 = y1 + CELL_PIXEL
                fill = CELL_BG if self.grid[r][c] == 0 else BLOCK_BG
                rect = self.canvas.create_rectangle(x1, y1, x2, y2, fill=fill, outline="#0f1a22")
                self.rects[r][c] = rect
                if self.grid[r][c] == 1:
                    txt = self.canvas.create_text(x1 + CELL_PIXEL/2, y1 + CELL_PIXEL/2,
                                                  text=BLOCK_EMOJI, font=("Segoe UI Emoji", int(CELL_PIXEL/2)))
                else:
                    txt = self.canvas.create_text(x1 + CELL_PIXEL/2, y1 + CELL_PIXEL/2,
                                                  text="", font=("Segoe UI Emoji", int(CELL_PIXEL/2)))
                self.texts[r][c] = txt

        for cp in self.checkpoints:
            r,c = cp
            self.canvas.itemconfigure(self.texts[r][c], text=CHECK_EMOJI)
            self.canvas.itemconfigure(self.rects[r][c], fill=CHECK_BG)

        gr, gc = self.goal
        self.canvas.itemconfigure(self.texts[gr][gc], text=GOAL_EMOJI)
        self.canvas.itemconfigure(self.rects[gr][gc], fill=GOAL_BG)

        px = self.player[1]*CELL_PIXEL + CELL_PIXEL/2
        py = self.player[0]*CELL_PIXEL + CELL_PIXEL/2
        self.player_obj = self.canvas.create_text(px, py, text=PLAYER_EMOJI, font=("Segoe UI Emoji", int(CELL_PIXEL/1.7)))

        self.master.bind("<Up>", lambda e: self.request_move(-1,0))
        self.master.bind("<Down>", lambda e: self.request_move(1,0))
        self.master.bind("<Left>", lambda e: self.request_move(0,-1))
        self.master.bind("<Right>", lambda e: self.request_move(0,1))
        self.master.bind("b", lambda e: self.bfs_hint())
        self.master.bind("a", lambda e: self.astar_assist())


        bottom = tk.Frame(self.master, bg=BG)
        bottom.pack(pady=6)
        tk.Button(bottom, text="BFS Hint (B)", command=self.bfs_hint, width=12).grid(row=0, column=0, padx=6)
        tk.Button(bottom, text="A* Auto (A)", command=self.astar_assist, width=12).grid(row=0, column=1, padx=6)
        tk.Button(bottom, text="Show Path", command=self.show_full_path, width=12).grid(row=0, column=2, padx=6)

    def update_info_labels(self):
        """Refresh runtime labels (steps, hints, level header)."""
        self.info_steps.config(text=f"Steps: {self.steps}")
        self.info_hints.config(text=f"Hints: {self.hints_left}")
        self.level_label.config(text=f"Level {self.level}  🎯 {len([c for c in self.checkpoints if c != (-1,-1)])} checkpoints    ⭐ Total: {self.total_stars}")

    def update_timer(self):
        """Update elapsed time every 0.5s."""
        if self.level_over:
            return
        elapsed = int(time.time() - (self.start_time or time.time()))
        self.info_timer.config(text=f"Time: {elapsed}s")
        self.master.after(500, self.update_timer)


    def request_move(self, dr, dc):
        """Request a single-step move from keyboard (if not animating)."""
        if self.level_over or self.animating:
            return
        nr = self.player[0] + dr
        nc = self.player[1] + dc
        if not (0 <= nr < GRID_SIZE and 0 <= nc < GRID_SIZE):
            return
        if self.grid[nr][nc] == 1:
            self.flash_cell(nr, nc)
            return
        self.animate_move_to(nr, nc)

    def flash_cell(self, r, c):
        """Briefly flash a blocked cell to indicate invalid move."""
        rect = self.rects[r][c]
        orig = self.canvas.itemcget(rect, "fill")
        self.canvas.itemconfigure(rect, fill=BLOCK_BG)
        self.master.after(150, lambda: self.canvas.itemconfigure(rect, fill=orig))

    def animate_move_to(self, nr, nc):
        """Smoothly animate the player to a target tile and handle post-move events."""
        self.animating = True
        start_x = self.player[1]*CELL_PIXEL + CELL_PIXEL/2
        start_y = self.player[0]*CELL_PIXEL + CELL_PIXEL/2
        target_x = nc*CELL_PIXEL + CELL_PIXEL/2
        target_y = nr*CELL_PIXEL + CELL_PIXEL/2
        dx = target_x - start_x
        dy = target_y - start_y
        steps = max(1, int(max(abs(dx), abs(dy)) / SMOOTH_STEP))
        step_x = dx / steps
        step_y = dy / steps
        i = 0

        def step():
            nonlocal i
            if i >= steps:
                
                self.canvas.coords(self.player_obj, target_x, target_y)

                pr, pc = self.player

                prev_is_goal = (pr, pc) == (self.goal[0], self.goal[1])
                prev_is_checkpoint = any(cp == (pr, pc) for cp in self.checkpoints)

                if not prev_is_checkpoint and not prev_is_goal:
                    self.canvas.itemconfigure(self.rects[pr][pc], fill=VISITED_BG)
                    self.canvas.itemconfigure(self.texts[pr][pc], text="")

                self.player = [nr, nc]

                for idx, cp in enumerate(self.checkpoints):
                    if cp == (nr, nc):
                        self.canvas.itemconfigure(self.rects[nr][nc], fill=CHECKED_BG)
                        self.canvas.itemconfigure(self.texts[nr][nc], text="✔")
                        self.checkpoints[idx] = (-1, -1)
                if (nr, nc) == (self.goal[0], self.goal[1]):
                    self.check_events()
                self.steps += 1
                self.update_info_labels()
                self.animating = False
                return

            self.canvas.move(self.player_obj, step_x, step_y)
            i += 1
            self.master.after(16, step) 

        step()

    def check_events(self):
        """Called whenever the player lands on a tile — handles goal/checkpoint validation."""
        pos = tuple(self.player)
        if pos == (self.goal[0], self.goal[1]):
            if any(cp != (-1,-1) for cp in self.checkpoints):
                gr, gc = self.goal
                self.canvas.itemconfigure(self.rects[gr][gc], fill=GOAL_BG)
                self.canvas.itemconfigure(self.texts[gr][gc], text=GOAL_EMOJI)
                messagebox.showinfo("Checkpoint", "Visit all 🎯 before collecting 💎!")
                return
            self.win()

    def win(self):
        """Compute shortest path (A* across original checkpoints), compute stars and show victory dialog."""
        self.level_over = True
        elapsed = int(time.time() - self.start_time) if self.start_time else 0

        remaining = [cp for cp in self.original_checkpoints if cp != (-1,-1)] + [self.goal]
        cur = tuple(self.start)
        full_shortest = []
        failed = False
        for wp in remaining:
            seg, _ = a_star_once(cur, wp, self.grid)
            if not seg:
                failed = True
                break
            if full_shortest and seg and seg[0] == full_shortest[-1]:
                full_shortest.extend(seg[1:])
            else:
                full_shortest.extend(seg)
            cur = wp

        shortest_moves = None if (failed or not full_shortest) else max(0, len(full_shortest) - 1)
        actual_moves = self.steps if self.steps > 0 else 0

        if actual_moves == 0:
            ratio = 1.0
        else:
            if shortest_moves is None:
                ratio = 0.0
            else:
                ratio = (shortest_moves / actual_moves)

        if ratio >= 0.95:
            stars_num = 5; stars_str = "⭐⭐⭐⭐⭐"
        elif ratio >= 0.85:
            stars_num = 4; stars_str = "⭐⭐⭐⭐"
        elif ratio >= 0.75:
            stars_num = 3; stars_str = "⭐⭐⭐"
        elif ratio >= 0.6:
            stars_num = 2; stars_str = "⭐⭐"
        else:
            stars_num = 1; stars_str = "⭐"

        self.total_stars += stars_num

        msg = (
            f"🎉 Level {self.level} Complete!\n"
            f"⭐ Rating: {stars_str}  ({stars_num} stars)\n"
            f"Steps Taken: {actual_moves}\n"
            f"Shortest Possible Moves: {shortest_moves if shortest_moves is not None else 'N/A'}\n"
            f"Time: {elapsed}s\n"
            f"Hints used: {HINTS_TOTAL - self.hints_left}\n\n"
            f"Total Stars (all levels): {self.total_stars}"
        )
        messagebox.showinfo("Victory!", msg)

    def bfs_hint(self):
        """Show a short BFS hint sequence — shuffle middle part to make hint look less optimal."""
        if self.level_over or self.animating:
            return
        if self.hints_left <= 0:
            messagebox.showwarning("No hints", "No hints left!")
            return

        remaining = [cp for cp in self.checkpoints if cp != (-1, -1)]
        target = remaining[0] if remaining else self.goal

        path, visited = bfs_once(tuple(self.player), target, self.grid)
        if not path:
            messagebox.showerror("No path", "BFS couldn't find a path to target.")
            return

        show_n = min(6, len(path))
        hint_cells = path[1:show_n]

        if len(hint_cells) > 3:
            middle = hint_cells[1:-1]
            random.shuffle(middle)
            hint_cells = [hint_cells[0]] + middle + [hint_cells[-1]]

        highlighted = []
        for (r, c) in hint_cells:
            if (r, c) == (self.goal[0], self.goal[1]):
                continue
            if any(cp == (r, c) for cp in self.checkpoints):
                continue
            self.canvas.itemconfigure(self.rects[r][c], fill=PATH_BG)
            self.canvas.itemconfigure(self.texts[r][c], text=PATH_EMOJI)
            highlighted.append((r, c))

        self.hints_left -= 1
        self.update_info_labels()
        self.master.after(HINT_SHOW_MS, lambda: self.clear_path_highlights(highlighted))

    def clear_path_highlights(self, cells_list):
        """Clear temporary path highlights and restore special cell visuals."""
        for (r,c) in cells_list:
            if (r,c) == (self.goal[0], self.goal[1]):
                self.canvas.itemconfigure(self.rects[r][c], fill=GOAL_BG)
                self.canvas.itemconfigure(self.texts[r][c], text=GOAL_EMOJI)
            elif any(cp == (r,c) for cp in self.checkpoints):
                self.canvas.itemconfigure(self.rects[r][c], fill=CHECK_BG)
                self.canvas.itemconfigure(self.texts[r][c], text=CHECK_EMOJI)
            else:
                self.canvas.itemconfigure(self.rects[r][c], fill=CELL_BG)
                self.canvas.itemconfigure(self.texts[r][c], text="")
    def astar_assist(self):
        """Automatically drive the player along an A* path to the next checkpoint or goal."""
        if self.level_over or self.animating:
            return
        if self.hints_left <= 0:
            messagebox.showwarning("No hints", "No hints left!")
            return

        remaining = [cp for cp in self.checkpoints if cp != (-1,-1)]
        target = remaining[0] if remaining else self.goal

        path, visited = a_star_once(tuple(self.player), target, self.grid)
        if not path:
            messagebox.showerror("No path", "A* couldn't find a path.")
            return

        self.hints_left -= 1
        self.update_info_labels()

        def step_along(idx):
            if idx >= len(path) or self.level_over:
                return
            nr, nc = path[idx]
            if self.grid[nr][nc] == 1:
                return
            def after_move_check():
                self.master.after(AUTO_STEP_DELAY_MS, lambda: step_along(idx+1))
            def move_and_wait():
                self.animate_move_to(nr, nc)
                def poll():
                    if self.animating:
                        self.master.after(20, poll)
                    else:
                        after_move_check()
                poll()
            move_and_wait()
        step_along(0)
    def show_full_path(self):
        path, _ = bfs_once(self.start, self.goal, self.grid)
        if not path:
            messagebox.showinfo("No path", "No path found to goal.")
            return
        highlights = []
        def paint(i):
            if i >= len(path):
                self.master.after(1000, lambda: self.clear_path_highlights(highlights))
                return
            r,c = path[i]
            if (r,c) != tuple(self.player) and (r,c) != (self.goal[0], self.goal[1]):
                self.canvas.itemconfigure(self.rects[r][c], fill=PATH_BG)
                self.canvas.itemconfigure(self.texts[r][c], text=PATH_EMOJI)
                highlights.append((r,c))
            self.master.after(60, lambda: paint(i+1))
        paint(0)
    def next_level(self):
        self.level += 1
        if hasattr(self, "canvas"):
            self.canvas.destroy()
        self.init_level()
if __name__ == "__main__":
    root = tk.Tk()
    root.geometry(f"{GRID_SIZE*CELL_PIXEL + 32}x{GRID_SIZE*CELL_PIXEL + 220}")
    root.minsize(600, 560)
    game = EmojiRushSmooth(root)
    root.mainloop()


In [1]:
import tkinter as tk
from tkinter import messagebox
import random
import heapq
import time
from collections import deque
import copy
import pygame

# --- Configuration ---
BASE_GRID_SIZE = 15
BLOCK_PROB = 0.28 
HINTS_TOTAL = 5
AUTO_STEP_DELAY_MS = 90
CELL_PIXEL = 50 
SMOOTH_STEP = 8
HINT_SHOW_MS = 1000
VOLUME_LEVEL = 0.15

PLAYER_EMOJI = "😎"
GOAL_EMOJI = "💎"
CHECK_EMOJI = "🎯"
BLOCK_EMOJI = "🔥"
PATH_EMOJI = "🟦"

BG = "#071020"
CELL_BG = "#08121a"
TEXT = "#e6f7ff"
GOAL_BG = "#ffd166"
CHECK_BG = "#c084fc"
PATH_BG = "#00e5ff"
BLOCK_BG = "#3b0b0b"
VISITED_BG = "#0b2a2a"
CHECKED_BG = "#064e3b" 

# --- Sound Manager ---
class SoundManager:
    def __init__(self):
        pygame.mixer.init()
        self.enabled = False
        try:
            self.step_snd = pygame.mixer.Sound("stepsound.wav")
            self.check_snd = pygame.mixer.Sound("checkpoint.wav")
            self.win_snd = pygame.mixer.Sound("applause.wav")
            for s in [self.step_snd, self.check_snd, self.win_snd]:
                s.set_volume(VOLUME_LEVEL)
            self.enabled = True
        except:
            pass

    def play_step(self):
        if self.enabled:
            self.step_snd.stop() 
            self.step_snd.play()
            self.step_snd.fadeout(120)

    def play_checkpoint(self):
        if self.enabled: self.check_snd.play()

    def play_win(self):
        if self.enabled: self.win_snd.play()

    def stop_win(self):
        if self.enabled: self.win_snd.fadeout(500)

# --- Pathfinding Helpers ---
def neighbors(r, c, grid, size):
    for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < size and 0 <= nc < size and grid[nr][nc] == 0:
            yield (nr, nc)

def reconstruct_path(came_from, end):
    path = []
    cur = end
    while cur in came_from:
        path.append(cur)
        cur = came_from[cur]
    path.reverse()
    return path

def bfs_once(start, goal, grid, size):
    start = tuple(start); goal = tuple(goal)
    q = deque([start])
    came_from = {}
    visited = {start}
    while q:
        node = q.popleft()
        if node == goal:
            return reconstruct_path(came_from, goal)
        for nb in neighbors(*node, grid, size):
            if nb not in visited:
                visited.add(nb)
                came_from[nb] = node
                q.append(nb)
    return None

def a_star_once(start, goal, grid, size):
    start = tuple(start); goal = tuple(goal)
    h = lambda a, b: abs(a[0]-b[0]) + abs(a[1]-b[1])
    open_set = [(0, start)]
    came_from = {}
    g = {start: 0}
    seen = set()
    while open_set:
        _, current = heapq.heappop(open_set)
        if current in seen: continue
        seen.add(current)
        if current == goal:
            return reconstruct_path(came_from, goal)
        for nb in neighbors(*current, grid, size):
            tentative = g[current] + 1
            if tentative < g.get(nb, float('inf')):
                came_from[nb] = current
                g[nb] = tentative
                heapq.heappush(open_set, (tentative + h(nb, goal), nb))
    return None

def carve_corridor(grid, a, b):
    (r1,c1),(r2,c2) = a,b
    r,c = r1,c1
    grid[r][c] = 0
    while r != r2:
        r += 1 if r2 > r else -1
        grid[r][c] = 0
    while c != c2:
        c += 1 if c2 > c else -1
        grid[r][c] = 0

def generate_solvable_grid(size, block_prob, checkpoints_count):
    while True:
        grid = [[1 if random.random() < block_prob else 0 for _ in range(size)] for _ in range(size)]
        free = [(r,c) for r in range(size) for c in range(size) if grid[r][c]==0]
        if len(free) < (checkpoints_count + 2): continue
        start = random.choice(free)
        free2 = [p for p in free if p != start]
        goal = random.choice(free2)
        random.shuffle(free2)
        cps = [p for p in free2 if p != start and p != goal][:checkpoints_count]
        waypoints = [start] + cps + [goal]
        for i in range(len(waypoints)-1):
            p = bfs_once(waypoints[i], waypoints[i+1], grid, size)
            if not p: carve_corridor(grid, waypoints[i], waypoints[i+1])
        return grid, start, cps, goal

# --- Main Game Class ---
class EmojiRushSmooth:
    def __init__(self, master):
        self.master = master
        master.title("Emoji Rush")
        master.config(bg=BG)
        self.sounds = SoundManager()
        self.level = 1
        self.total_stars = 0
        self.grid_size = BASE_GRID_SIZE
        self.init_level()

    def init_level(self):
        self.sounds.stop_win()
        if self.level >= 30: self.grid_size = 25
        elif self.level >= 15: self.grid_size = 20
        else: self.grid_size = BASE_GRID_SIZE

        level_checkpoints = self.level
        self.grid, self.start, self.checkpoints, self.goal = generate_solvable_grid(self.grid_size, BLOCK_PROB, level_checkpoints)
        self.original_checkpoints = copy.deepcopy(self.checkpoints)
        self.player = list(self.start)
        self.hints_left = HINTS_TOTAL
        self.steps = 0
        self.level_over = self.animating = False
        self.start_time = time.time()
        
        win_w = self.grid_size * CELL_PIXEL + 40
        win_h = self.grid_size * CELL_PIXEL + 220
        self.master.geometry(f"{win_w}x{win_h}")
        self.build_ui()
        self.update_timer()

    def build_ui(self):
        for w in self.master.winfo_children(): w.destroy()
        top_container = tk.Frame(self.master, bg=BG)
        top_container.pack(pady=10, fill="x")

        self.level_label = tk.Label(
            top_container, 
            text=f"Level {self.level} | Total Stars: {self.total_stars} | Target: {len(self.original_checkpoints)} 🎯", 
            font=("Helvetica", 14, "bold"), fg=TEXT, bg=BG
        )
        self.level_label.pack(side="top")

        stats_frame = tk.Frame(top_container, bg=BG)
        stats_frame.pack(side="top", pady=5)

        self.info_steps = tk.Label(stats_frame, text=f"Steps: {self.steps}", fg=TEXT, bg=BG)
        self.info_steps.pack(side="left", padx=10)
        self.info_hints = tk.Label(stats_frame, text=f"Hints: {self.hints_left}", fg=TEXT, bg=BG)
        self.info_hints.pack(side="left", padx=10)
        self.info_timer = tk.Label(stats_frame, text="Time: 0s", fg=TEXT, bg=BG)
        self.info_timer.pack(side="left", padx=10)
        tk.Button(stats_frame, text="Next Level", command=self.next_level, width=10).pack(side="left", padx=10)

        self.canvas = tk.Canvas(self.master, width=self.grid_size*CELL_PIXEL, height=self.grid_size*CELL_PIXEL, bg=BG, highlightthickness=0)
        self.canvas.pack(padx=8, pady=6)

        self.rects = [[None]*self.grid_size for _ in range(self.grid_size)]
        self.texts = [[None]*self.grid_size for _ in range(self.grid_size)]
        
        for r in range(self.grid_size):
            for c in range(self.grid_size):
                x1, y1 = c*CELL_PIXEL, r*CELL_PIXEL
                fill = CELL_BG if self.grid[r][c] == 0 else BLOCK_BG
                self.rects[r][c] = self.canvas.create_rectangle(x1, y1, x1+CELL_PIXEL, y1+CELL_PIXEL, fill=fill, outline="#0f1a22")
                txt = BLOCK_EMOJI if self.grid[r][c] == 1 else ""
                self.texts[r][c] = self.canvas.create_text(x1+CELL_PIXEL/2, y1+CELL_PIXEL/2, text=txt, font=("Segoe UI Emoji", 18))

        for r, c in self.checkpoints:
            self.canvas.itemconfigure(self.rects[r][c], fill=CHECK_BG)
            self.canvas.itemconfigure(self.texts[r][c], text=CHECK_EMOJI)

        gr, gc = self.goal
        self.canvas.itemconfigure(self.rects[gr][gc], fill=GOAL_BG)
        self.canvas.itemconfigure(self.texts[gr][gc], text=GOAL_EMOJI)

        px, py = self.player[1]*CELL_PIXEL+CELL_PIXEL/2, self.player[0]*CELL_PIXEL+CELL_PIXEL/2
        self.player_obj = self.canvas.create_text(px, py, text=PLAYER_EMOJI, font=("Segoe UI Emoji", 24))

        bottom = tk.Frame(self.master, bg=BG)
        bottom.pack(pady=10)
        tk.Button(bottom, text="BFS Hint (B)", command=self.bfs_hint, width=12).pack(side="left", padx=5)
        tk.Button(bottom, text="A* Auto (A)", command=self.astar_assist, width=12).pack(side="left", padx=5)
        tk.Button(bottom, text="Show Path", command=self.show_full_path, width=12).pack(side="left", padx=5)

        self.master.bind("<Up>", lambda e: self.request_move(-1,0))
        self.master.bind("<Down>", lambda e: self.request_move(1,0))
        self.master.bind("<Left>", lambda e: self.request_move(0,-1))
        self.master.bind("<Right>", lambda e: self.request_move(0,1))

    def request_move(self, dr, dc):
        if self.level_over or self.animating: return
        nr, nc = self.player[0]+dr, self.player[1]+dc
        if 0 <= nr < self.grid_size and 0 <= nc < self.grid_size and self.grid[nr][nc] == 0:
            self.sounds.play_step()
            self.animate_move_to(nr, nc)

    def animate_move_to(self, nr, nc):
        self.animating = True
        tx, ty = nc*CELL_PIXEL+CELL_PIXEL/2, nr*CELL_PIXEL+CELL_PIXEL/2
        sx, sy = self.player[1]*CELL_PIXEL+CELL_PIXEL/2, self.player[0]*CELL_PIXEL+CELL_PIXEL/2
        dx, dy = (tx-sx)/SMOOTH_STEP, (ty-sy)/SMOOTH_STEP
        
        def step(i=0):
            if i < SMOOTH_STEP:
                self.canvas.move(self.player_obj, dx, dy)
                self.master.after(16, lambda: step(i+1))
            else:
                pr, pc = self.player
                if (pr, pc) not in self.original_checkpoints and (pr, pc) != self.goal:
                    self.canvas.itemconfigure(self.rects[pr][pc], fill=VISITED_BG)
                self.player = [nr, nc]
                for idx, cp in enumerate(self.checkpoints):
                    if cp == (nr, nc):
                        self.sounds.play_checkpoint()
                        self.canvas.itemconfigure(self.rects[nr][nc], fill=CHECKED_BG)
                        self.canvas.itemconfigure(self.texts[nr][nc], text="✔")
                        self.checkpoints[idx] = (-1, -1)
                self.steps += 1
                self.update_info_labels()
                self.animating = False
                if (nr, nc) == self.goal: self.check_events()
        step()

    def check_events(self):
        remaining = [cp for cp in self.checkpoints if cp != (-1, -1)]
        if remaining:
            messagebox.showinfo("Wait!", f"Collect {len(remaining)} more 🎯!")
            return
        self.win()

    def get_optimal_path_count(self):
        """FIXED: Absolutley shortest path using pristine grid and nearest neighbor sorting."""
        # 1. PRISTINE GRID: Only fire blocks are walls
        clean_grid = [[1 if self.grid[r][c] == 1 else 0 for c in range(self.grid_size)] for r in range(self.grid_size)]
        
        total_optimal = 0
        current_pos = tuple(self.start)
        pending_checkpoints = [cp for cp in self.original_checkpoints if cp != (-1, -1)]
        
        # 2. NEAREST NEIGHBOR: Always find the closest target next to find the true minimum
        while pending_checkpoints:
            best_dist = float('inf')
            best_target = None
            best_path_len = 0
            
            for cp in pending_checkpoints:
                path = a_star_once(current_pos, cp, clean_grid, self.grid_size)
                if path and (len(path) - 1) < best_dist:
                    best_dist = len(path) - 1
                    best_target = cp
                    best_path_len = len(path) - 1
            
            if best_target:
                total_optimal += best_path_len
                current_pos = best_target
                pending_checkpoints.remove(best_target)
            else:
                break # Should not happen on solvable grid
        
        # 3. FINAL GOAL
        final_path = a_star_once(current_pos, self.goal, clean_grid, self.grid_size)
        if final_path:
            total_optimal += len(final_path) - 1
            
        return total_optimal

    def win(self):
        self.level_over = True
        self.sounds.play_win()
        shortest_moves = self.get_optimal_path_count() # Recalculate correctly
        
        ratio = shortest_moves / self.steps if self.steps > 0 else 1.0
        stars = 5 if ratio >= 0.9 else 4 if ratio >= 0.75 else 3 if ratio >= 0.6 else 2 if ratio >= 0.4 else 1
        self.total_stars += stars
        
        messagebox.showinfo("Victory!", f"🎉 Level {self.level} Clear!\n⭐ Rating: {'⭐'*stars}\nSteps: {self.steps}\nOptimal: {shortest_moves}")

    def bfs_hint(self):
        if self.hints_left <= 0 or self.level_over: return
        target = [cp for cp in self.checkpoints if cp != (-1, -1)]
        target = target[0] if target else self.goal
        path = bfs_once(self.player, target, self.grid, self.grid_size)
        if path:
            self.hints_left -= 1
            h_cells = path[1:min(6, len(path))]
            for r, c in h_cells:
                self.canvas.itemconfigure(self.rects[r][c], fill=PATH_BG)
            self.master.after(HINT_SHOW_MS, lambda: self.clear_path_highlights(h_cells))
            self.update_info_labels()

    def clear_path_highlights(self, cells):
        for r, c in cells:
            if (r,c) == self.goal: fill = GOAL_BG
            elif any(cp == (r,c) for cp in self.original_checkpoints): fill = CHECK_BG
            else: fill = CELL_BG
            self.canvas.itemconfigure(self.rects[r][c], fill=fill)

    def astar_assist(self):
        if self.hints_left <= 0 or self.level_over or self.animating: return
        target = [cp for cp in self.checkpoints if cp != (-1, -1)]
        target = target[0] if target else self.goal
        path = a_star_once(self.player, target, self.grid, self.grid_size)
        if path:
            self.hints_left -= 1
            def step_along(idx):
                if idx < len(path) and not self.level_over:
                    self.animate_move_to(path[idx][0], path[idx][1])
                    self.master.after(200, lambda: step_along(idx+1))
            step_along(1)

    def show_full_path(self):
        path = bfs_once(self.player, self.goal, self.grid, self.grid_size)
        if path:
            for r,c in path[1:-1]: self.canvas.itemconfigure(self.rects[r][c], fill=PATH_BG)
            self.master.after(HINT_SHOW_MS, lambda: self.clear_path_highlights(path[1:-1]))

    def update_info_labels(self):
        self.info_steps.config(text=f"Steps: {self.steps}")
        self.info_hints.config(text=f"Hints: {self.hints_left}")
        self.level_label.config(text=f"Level {self.level} | Total Stars: {self.total_stars} | Target: {len(self.original_checkpoints)} 🎯")

    def update_timer(self):
        if not self.level_over:
            self.info_timer.config(text=f"Time: {int(time.time() - self.start_time)}s")
            self.master.after(1000, self.update_timer)

    def next_level(self):
        self.level += 1; self.init_level()

if __name__ == "__main__":
    root = tk.Tk()
    game = EmojiRushSmooth(root)
    root.mainloop()

<frozen importlib._bootstrap>:488: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.


pygame 2.5.2 (SDL 2.30.0, Python 3.12.3)
Hello from the pygame community. https://www.pygame.org/contribute.html
